In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;
typedef std::pair<float, int> Entry; // coppia (u, indice) per la coda di priorita'

__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    // Per arrivare devi scorrere come un indice globale: u[i][j][k] = u[(k * N + i) * N + j]
    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, const std::vector<int>& srcs, const std::vector<unsigned char>& mask)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    for (int s : srcs) {                             // tutte le sorgenti sono i primi Trial
        heap.push(Entry(u[s], s));
        state[s] = ATTIVO;
    }

    const int dk[6] = { 0, 0, 0, 0, -1, 1 };              // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1, 0, 0 };              // spostamenti verso i 6 vicini:
    const int dj[6] = { -1, 1,  0, 0, 0, 0 };              // sinistra, destra, sotto, sopra

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO || !mask[nb]) continue;         // già noto, oppure ostacolo

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
template <class Esatta>
double relative_error(const std::vector<float>& u, int N, float h, Esatta esatta)
{
    double num = 0.0, den = 0.0;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h, z = -1.0 + k * h;   // coordinate del nodo (k, i, j)
            double U = esatta(x, y, z);                         // soluzione esatta
            double d = u[(k * N + i) * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
            }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}


In [ ]:
%%writefile ambiente.h
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1
struct Sfera   { double cx, cy, cz, r; };                 // centro (cx, cy, cz) e raggio r

// =====================================================================
// L'AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.85, -0.85, -0.85 };                 // partenza del drone: angolo in basso
const Punto B = {  0.85,  0.85,  0.85 };                 // arrivo (sorgente del FIM): angolo opposto, in alto

const std::vector<Scatola> SCATOLE = {                   // {x0, x1, y0, y1, z0, z1}
    { -0.45, -0.35, -1.0, 1.0, -1.0, 1.0 },              // muro 1 (x = -0.4)
    {  0.15,  0.25, -1.0, 1.0, -1.0, 1.0 },              // muro 2 (x = 0.2)
    {  0.45,  0.65,  0.2, 1.0,  0.2, 1.0 },              // blocco davanti a B: si entra solo da sotto o di lato
    { -1.0,  -0.45, -0.3, -0.2, -1.0, 0.4 },             // paratia tra A e il muro 1
    { -1.0,  -0.7,  -1.0, -0.3, -0.2, -0.1 },            // mensola sopra A: si sale dal varco vicino al muro 1
    { -0.15, -0.05, -0.05, 0.05, -1.0, 1.0 },            // pilastro tra i due muri, da terra al soffitto
    {  0.25,  1.0,  -0.45, -0.35, 0.3, 1.0 },            // trave appesa al soffitto dopo il muro 2: si passa sotto
};
const std::vector<Sfera> SFERE = {                       // {cx, cy, cz, r}
    { -0.1,  0.3, -0.4, 0.25 },                          // sfera tra i due muri, davanti alla finestra 1
    { -0.1, -0.4,  0.4, 0.22 },                          // sfera tra i due muri, davanti alla finestra 2
    {  0.6, -0.1, -0.1, 0.2 },                           // sfera dopo il muro 2, sotto la trave
};
const std::vector<Scatola> FORI = {                      // zone in cui l'ostacolo viene tolto
    { -0.45, -0.35,  0.5,  0.8, -0.8, -0.5 },            // finestra nel muro 1: y alta, z bassa
    {  0.15,  0.25, -0.8, -0.5,  0.5,  0.8 },            // finestra nel muro 2: y bassa, z alta
};
// =====================================================================

inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

inline bool dentro(const Sfera& s, double x, double y, double z)
{
    return (x - s.cx) * (x - s.cx) + (y - s.cy) * (y - s.cy) + (z - s.cz) * (z - s.cz) <= s.r * s.r;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    for (const auto& s : SFERE)   if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia N^3: 1 = nodo libero, 0 = nodo ostacolo (resta INF), margine compreso
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    std::vector<unsigned char> mask((size_t)N * N * N);
    for (int k = 0; k < N; ++k)                              // 1) gli ostacoli veri
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j)
                mask[((size_t)k * N + i) * N + j] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;

    // 2) Margine di sicurezza: diventa ostacolo ogni nodo a distanza <= MARGINE da un nodo ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio del margine, in nodi
    std::vector<unsigned char> allargata = mask;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                if (mask[((size_t)k * N + i) * N + j]) continue;                       // si parte solo dai nodi ostacolo...
                bool bordo = false;                        // ...che toccano un nodo libero
                if (j > 0     && mask[((size_t)k * N + i) * N + j - 1]) bordo = true;
                if (j < N - 1 && mask[((size_t)k * N + i) * N + j + 1]) bordo = true;
                if (i > 0     && mask[((size_t)k * N + i - 1) * N + j]) bordo = true;
                if (i < N - 1 && mask[((size_t)k * N + i + 1) * N + j]) bordo = true;
                if (k > 0     && mask[((size_t)(k - 1) * N + i) * N + j]) bordo = true;
                if (k < N - 1 && mask[((size_t)(k + 1) * N + i) * N + j]) bordo = true;
                if (!bordo) continue;
                for (int dk = -r; dk <= r; ++dk)                  // tutti i nodi entro la sfera di raggio MARGINE
                    for (int di = -r; di <= r; ++di)
                        for (int dj = -r; dj <= r; ++dj) {
                            const int kk = k + dk, ii = i + di, jj = j + dj;
                            if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                            if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                                allargata[((size_t)kk * N + ii) * N + jj] = 0;
                        }
            }
    return allargata;
}

// le sorgenti sono i nodi liberi che toccano un nodo ostacolo (il bordo degli ostacoli)
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// dalla distanza dagli ostacoli alla lentezza F = 1/v, con v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1)
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: r = 0 sul bordo (F = 1/V_MIN = 10, massima), r = 1 oltre D_SICUREZZA o se d = INF (F = 1)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}

inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

In [ ]:
%%writefile fim3d.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include "fmm3d.h"           // Fast Marching
#include "ambiente.h"       // Ostacoli

// Controllo degli errori CUDA: se la chiamata fallisce, stampa file, riga e motivo ed esce.
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // dimensione del blocco 8x8x8 = 512 thread
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

__device__ void process_block(int block, int niter, const float* u_in, float* u_out, const float* F, const unsigned char* mask,
                             int N, int nbx, float h, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];                // block + halo, nella shared memory
    __shared__ unsigned char s_c[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);            // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * BLOCK + ty) * BLOCK + tx;                                       // numero del thread nel blocco: 0..511
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;                                                  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // fuori dominio non aggiornare nodi
    float f = inside ? F[idx] : 1.0f;                  // F_ijk del nodo

    // Global -> Shared: 1000 celle (block + halo), 512 thread
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        int sz = m / ((BLOCK + 2) * (BLOCK + 2));
        int sy = (m / (BLOCK + 2)) % (BLOCK + 2);
        int sx = m % (BLOCK + 2);                             // cella (sz, sy, sx) della shared memory
        int gi = bi * BLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * BLOCK + sx - 1;                         
        int gk = bk * BLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u_in[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la block e' completa prima di usarla

    // niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                 // valore prima dell'aggiornamento
        float q = fixed ? p                                  // fuori dominio: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // Shared -> Global: ogni thread riscrive il PROPRIO nodo (non l'halo)
    if (inside) u_out[idx] = s_u[tz+1][ty + 1][tx + 1];

    // Cb(b) <- reduction(Cn(b)): AND dei 512 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = BLOCK * BLOCK * BLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

__global__ void update_active_blocks(const int* list, const float* u_in, float* u_out, const float* F, const unsigned char* mask,
                                    int N, int nbx, float h, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u_in, u_out, F, mask, N, nbx, h, eps, Cb1);
}

__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco di nodi
    if (t >= n_blocks) return;                               // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A
}

__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // blockDim = 256
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        const float* u_in, float* u_out, const float* F, const unsigned char* mask, int N, int nbx, float h, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u_in, u_out, F, mask, N, nbx, h, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}
__global__ void copy_blocks(const int* list, const int* count, const float* u2, float* u, int N, int nbx)
{
    if (blockIdx.x >= *count) return;                        // blocco CUDA in eccesso
    int b = list[blockIdx.x];
    int bi = (b / nbx) % nbx, bj = b % nbx, bk = b / (nbx * nbx);
    int i = bi * BLOCK + threadIdx.y, j = bj * BLOCK + threadIdx.x, k = bk * BLOCK + threadIdx.z;
    if (i < N && j < N && k < N) u[(k * N + i) * N + j] = u2[(k * N + i) * N + j];
}

int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx; // nbx = numero di blocchi per riga/colonna, n_blocks = numero totale di blocchi
    const size_t bytes = (size_t)N * N * N * sizeof(float); // dimensione in byte della griglia

    std::vector<unsigned char> Cb0(n_blocks, 1);    // flag Cb iniziali: 1 = convergente, 0 = attivo
    // I blocchi che contengono una sorgente partono attivi Cb0 = 0
    for (int s : srcs) {
        const int sk = s / (N * N), si = (s / N) % N, sj = s % N;   // strato, riga e colonna del nodo sorgente
        Cb0[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;   // il blocco di questa sorgente parte attivo
    }

    // Memoria sulla GPU e dati iniziali
    float *d_u, *d_u2, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count; unsigned char* d_mask;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_u2, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_mask, (size_t)N * N * N));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Memoria per i blocchi candidati (vicini dei blocchi appena convergenti)
    CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_u2, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), (size_t)N * N * N, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));


    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;   // numero di blocchi per costruire la lista A
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo
    int iter = 0;
    while (true) {
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_active, d_list, d_count, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (count == 0) break;  // se count == 0, lista A vuota, termina ciclo
        ++iter;
        // Step 1: foreach b in A in parallelo
        update_active_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_u, d_u2, d_F, d_mask, N, nbx, h, niter, eps, d_Cb1);    // si lanciano count blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        copy_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_count, d_u2, d_u, N, nbx);
        CUDA_CHECK(cudaGetLastError());
        // Step 2: solo i vicini dei blocchi appena convergenti
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        build_candidate_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                          d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                    d_u, d_u2, d_F, d_mask, N, nbx, h, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
        copy_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_u2, d_u, N, nbx);
        CUDA_CHECK(cudaGetLastError());
        }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sull'host e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_u2)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = BLOCK;                              // 8 iterazioni locali
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure
    
     if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("FM2: passata 1 = distanza dagli ostacoli, passata 2 = tempo fino a B con F lenta vicino agli ostacoli\n");
    printf("    N   giri   max|FIM - FMM|   u(A) FIM   u(A) FMM   NaN   ostacoli non INF   t FMM [ms]   t FIM [ms]   speedup | passata 1: giri   max|FIM - FMM|   t FMM [ms]   t FIM [ms] | speedup FM2\n");

    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // griglia (§2.4)
        const int n = Nk * Nk * Nk;                          // numero di nodi
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0 = ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // la sorgente e' l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        // passata 1: distanza dagli ostacoli (sorgenti = nodi liberi sul bordo degli ostacoli, F = 1), con FIM e con FMM
        const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
        std::vector<float> d0(n, INF);                       // condizione iniziale: 0 sul bordo, INF altrove
        for (int s : bordo) d0[s] = 0.0f;
        const std::vector<float> uno(n, 1.0f);               // F = 1: u = distanza
        std::vector<float> d_ost, d_fmm;
        double t1_fmm = 1e30, t1_fim = 1e30;
        int it1 = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            d_fmm = d0;
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(d_fmm, uno, Nk, hk, bordo, mask);            // CPU: passata 1 con il FMM
            auto t1 = std::chrono::high_resolution_clock::now();
            t1_fmm = std::min(t1_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());

            d_ost = d0;
            float ms = 0.0f;
            it1 = block_fim(d_ost, uno, Nk, hk, bordo, mask, eps, niter, &ms);   // GPU: passata 1 con il FIM
            t1_fim = std::min(t1_fim, (double)ms);
        }
        float diff1 = 0.0f;                                  // la distanza del FIM deve coincidere con quella del FMM
        for (int q = 0; q < n; ++q) diff1 = fmaxf(diff1, fabsf(d_ost[q] - d_fmm[q]));
    

        // la lentezza F viene dalla distanza dagli ostacoli (lento vicino ai muri, F = 1 lontano); passata 2 qui sotto
        const std::vector<float> Fk = lentezza_fm2(d_ost);
        std::vector<float> u0(n, INF);                       // condizione iniziale
        for (int s : srcs) u0[s] = 0.0f;

        std::vector<float> uk_fmm, uk_fim;
        double t_fmm = 1e30, t_fim = 1e30;
        int it = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            uk_fmm = u0;                                     // ogni misura riparte da capo
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(uk_fmm, Fk, Nk, hk, srcs, mask);             // CPU: Algoritmo 3
            auto t1 = std::chrono::high_resolution_clock::now();
            t_fmm = std::min(t_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());

            uk_fim = u0;
            float ms = 0.0f;
            it = block_fim(uk_fim, Fk, Nk, hk, srcs, mask, eps, niter, &ms);   // GPU: Algoritmo 7
            t_fim = std::min(t_fim, (double)ms);
        }

        // Controlli: FIM = FMM, nessun NaN, gli ostacoli restano INF
        float diff = 0.0f;
        int n_nan = 0, ost_non_inf = 0;
        for (int q = 0; q < n; ++q) {
            diff = fmaxf(diff, fabsf(uk_fim[q] - uk_fmm[q]));
            if (std::isnan(uk_fim[q])) ++n_nan;
            if (!mask[q] && uk_fim[q] < INF) ++ost_non_inf;
        }
        int non_raggiunti = 0;                               // nodi liberi che l'onda non ha raggiunto
        for (int q = 0; q < n; ++q) if (mask[q] && uk_fim[q] >= INF) ++non_raggiunti;
        if (uk_fim[iA] >= INF)                               // l'onda partita da B non e' mai arrivata ad A
            printf("%5d   %4d   %14.2e   A NON RAGGIUNGIBILE da B (nodi liberi non raggiunti: %d)\n", Nk, it, diff, non_raggiunti);
        else
            printf("%5d   %4d   %14.2e   %8.4f   %8.4f   %3d   %16d   %10.1f   %10.2f   %7.1f | %15d   %14.2e   %10.1f   %10.2f | %11.1f\n", Nk, it, diff,
                   uk_fim[iA], uk_fmm[iA], n_nan, ost_non_inf, t_fmm, t_fim, t_fmm / t_fim,
                   it1, diff1, t1_fmm, t1_fim, (t1_fmm + t_fmm) / (t1_fim + t_fim));

        if (Nk == 65) {                                      // per i grafici (Python): soluzione, maschera e ambiente
            std::ofstream out("fim_65_ostacoli.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(uk_fim.data()), uk_fim.size() * sizeof(float));
            std::ofstream om("mask_65.bin", std::ios::binary);
            om.write(reinterpret_cast<const char*>(mask.data()), mask.size());
            std::ofstream amb("ambiente.txt");               // l'ambiente di ambiente.h, una riga per elemento
            amb.precision(15);                               // 15 cifre: i numeri arrivano a Python esattamente come in ambiente.h
            amb << "MARGINE " << MARGINE << "\n";
            amb << "A " << A.x << " " << A.y << " " << A.z << "\n";
            amb << "B " << B.x << " " << B.y << " " << B.z << "\n";
            for (const auto& s : SCATOLE) amb << "SCATOLA " << s.x0 << " " << s.x1 << " " << s.y0 << " " << s.y1 << " " << s.z0 << " " << s.z1 << "\n";
            for (const auto& s : SFERE)   amb << "SFERA " << s.cx << " " << s.cy << " " << s.cz << " " << s.r << "\n";
            for (const auto& f : FORI)    amb << "FORO " << f.x0 << " " << f.x1 << " " << f.y0 << " " << f.y1 << " " << f.z0 << " " << f.z1 << "\n";
            std::ofstream of("F_65.bin", std::ios::binary);  // la lentezza F usata nella passata 2 (serve a Python)
            of.write(reinterpret_cast<const char*>(Fk.data()), Fk.size() * sizeof(float));
        }
    }
    
    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim3d.cu -o fim3d && ./fim3d

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial import cKDTree
from skimage.measure import marching_cubes

# =====================================================================
# 1) DATI scritti dal programma C++ (N = 65): soluzione del FIM, maschera, lentezza F, ambiente (definito solo in ambiente.h)
# =====================================================================
N = 65
h = 2.0 / (N - 1)
u = np.fromfile("fim_65_ostacoli.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)   # u[k, i, j] = u(z, y, x)
mask = np.fromfile("mask_65.bin", dtype=np.uint8).reshape(N, N, N)                             # 1 = libero, 0 = ostacolo
libero = mask == 1
x = -1.0 + h * np.arange(N)
Fl = np.fromfile("F_65.bin", dtype=np.float32).reshape(N, N, N).astype(np.float64)              # lentezza F usata dal FIM

amb = {"SCATOLA": [], "SFERA": [], "FORO": []}
for riga in open("ambiente.txt"):
    nome, *valori = riga.split()
    valori = [float(v) for v in valori]
    if nome == "A":         A = np.array(valori)
    elif nome == "B":       B = np.array(valori)
    elif nome == "MARGINE": MARGINE = valori[0]
    else:                   amb[nome].append(tuple(valori))
SCATOLE, SFERE, FORI = amb["SCATOLA"], amb["SFERA"], amb["FORO"]

def nodo(p):                                                        # nodo (k, i, j) piu' vicino al punto p = (x, y, z)
    return tuple(np.round((np.asarray(p)[::-1] + 1.0) / h).astype(int))
def coord(n):                                                       # coordinate (x, y, z) del nodo n = (k, i, j)
    return -1.0 + h * np.array(n[::-1], dtype=float)

if u[nodo(A)] >= 1e29:                                              # u(A) = INF: l'onda partita da B non e' mai arrivata ad A
    raise SystemExit("A non e' raggiungibile da B: nessun percorso (controlla ostacoli e fori in ambiente.h)")

# =====================================================================
# 2) PERCORSO da A a B (il piu' veloce secondo u)
# =====================================================================
# Discesa sui nodi: da A si passa al vicino libero (tra i 26 attorno) da cui si arriva prima a B: u del vicino + tempo per raggiungerlo
mosse = [(dk, di, dj) for dk in (-1, 0, 1) for di in (-1, 0, 1) for dj in (-1, 0, 1) if (dk, di, dj) != (0, 0, 0)]
n, nB = nodo(A), nodo(B)
nodi = [n]
while n != nB:
    vicini = [(n[0] + a, n[1] + b, n[2] + c) for a, b, c in mosse]
    vicini = [m for m in vicini if all(0 <= t < N for t in m) and libero[m]]
    n = min(vicini, key=lambda m: u[m] + 0.5 * (Fl[n] + Fl[m]) * h * np.linalg.norm(np.subtract(m, n)))   # u del vicino + tempo per arrivarci
    nodi.append(n)

# Accorciamento: da ogni punto si salta al nodo piu' avanti che si "vede" senza attraversare ostacoli
# e che non fa perdere tempo (tempo = integrale di F lungo il tratto, come nell'iconale)
peso = RegularGridInterpolator((x, x, x), libero.astype(float))     # = 1 solo se gli 8 nodi attorno sono liberi
F_interp = RegularGridInterpolator((x, x, x), Fl)
def tempo(p, q):                                                    # tempo lungo il segmento p-q: somma di F * ds
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    medi = p + (t[:-1] + t[1:]) / 2 * (q - p)
    return np.sum(F_interp(medi[:, ::-1])) * np.linalg.norm(q - p) / (len(t) - 1)
def si_vede(p, q):
    t = np.linspace(0.0, 1.0, int(np.linalg.norm(q - p) / (h / 4)) + 2)[:, None]
    return np.all(peso((p + t * (q - p))[:, ::-1]) > 0.999)
P = [coord(m) for m in nodi]
t_nodi = np.concatenate([[0], np.cumsum([tempo(P[i], P[i + 1]) for i in range(len(P) - 1)])])   # tempo lungo la discesa
percorso, a = [P[0]], 0
while a < len(P) - 1:
    b = len(P) - 1
    while b > a + 1 and not (si_vede(P[a], P[b]) and tempo(P[a], P[b]) <= t_nodi[b] - t_nodi[a]):
        b -= 1
    percorso.append(P[b])
    a = b
percorso = np.array(percorso)
lunghezza = np.sum(np.linalg.norm(np.diff(percorso, axis=0), axis=1))
tempo_percorso = sum(tempo(percorso[i], percorso[i + 1]) for i in range(len(percorso) - 1))

# =====================================================================
# 3) VERIFICA NUMERICA con la geometria esatta degli ostacoli (indipendente dalla griglia del FIM)
# =====================================================================
def dentro(b, p):
    return (b[0] <= p[:, 0]) & (p[:, 0] <= b[1]) & (b[2] <= p[:, 1]) & (p[:, 1] <= b[3]) & (b[4] <= p[:, 2]) & (p[:, 2] <= b[5])
def ostacolo_vero(p):                                               # vero per i punti p (righe x, y, z) dentro un ostacolo
    occ = np.zeros(len(p), bool)
    for s in SCATOLE: occ |= dentro(s, p)
    for c in SFERE:   occ |= np.linalg.norm(p - np.array(c[:3]), axis=1) <= c[3]
    for f in FORI:    occ &= ~dentro(f, p)
    return occ

fitto = np.concatenate([p + np.linspace(0, 1, 200, endpoint=False)[:, None] * (q - p)
                        for p, q in zip(percorso[:-1], percorso[1:])] + [percorso[-1:]])   # 200 punti per segmento
g = np.linspace(-1, 1, 201)                                         # griglia fine (passo 0.01) per la distanza dagli ostacoli
G = np.stack(np.meshgrid(g, g, g, indexing="ij")[::-1], axis=-1).reshape(-1, 3)
dist, _ = cKDTree(G[ostacolo_vero(G)]).query(fitto)

print(f"u(A) = {u[nodo(A)]:.4f}   tempo lungo il percorso = {tempo_percorso:.4f}   (devono quasi coincidere)")
print(f"lunghezza del percorso = {lunghezza:.4f}   linea retta |A - B| = {np.linalg.norm(A - B):.4f}")
print(f"punti del percorso controllati: {len(fitto)}, dentro un ostacolo: {ostacolo_vero(fitto).sum()}")
print(f"distanza minima del percorso dagli ostacoli: {dist.min():.3f}  (MARGINE = {MARGINE})")
for i, f in enumerate(FORI):
    sel = dentro(f, fitto)
    print(f"foro {i + 1}: " + (f"attraversato in ({fitto[sel, 0].mean():.2f}, {fitto[sel, 1].mean():.2f}, {fitto[sel, 2].mean():.2f})"
                               if sel.any() else "non usato"))

# =====================================================================
# 4) VISTA 3D INTERATTIVA: si ruota e si zooma con il mouse; lo slider fa avanzare il fronte d'onda partito da B
# =====================================================================
Pn = np.stack(np.meshgrid(x, x, x, indexing="ij")[::-1], axis=-1).reshape(-1, 3)
pieno = ostacolo_vero(Pn).reshape(N, N, N)                          # ostacoli veri sulla griglia del FIM
vo, fo, _, _ = marching_cubes(np.pad(pieno.astype(float), 1), level=0.5, spacing=(h, h, h))
po = vo - h - 1.0                                                   # vertici in ordine (z, y, x)
ostacoli = go.Mesh3d(x=po[:, 2], y=po[:, 1], z=po[:, 0], i=fo[:, 0], j=fo[:, 1], k=fo[:, 2],
                     color="lightgray", opacity=0.35, name="ostacoli", flatshading=True)
s_perc = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(percorso, axis=0), axis=1))])
linea = go.Scatter3d(x=percorso[:, 0], y=percorso[:, 1], z=percorso[:, 2], mode="lines+markers", name="percorso",
                     line=dict(color=s_perc[-1] - s_perc, colorscale="Reds_r", width=9), marker=dict(size=2, color="darkred"))
punti = go.Scatter3d(x=[A[0], B[0]], y=[A[1], B[1]], z=[A[2], B[2]], mode="markers+text", text=["A", "B"],
                     marker=dict(size=7, color=["green", "blue"]), textfont=dict(size=16), name="A e B")

w = np.where(libero, u, u[libero].max() * 1.05)                     # negli ostacoli un valore alto ma finito
istanti = np.linspace(u[libero].max() * 0.04, u[libero].max() * 0.98, 15)
def fronte(t):
    v, f, _, _ = marching_cubes(w, level=t, spacing=(h, h, h), step_size=2)
    p = v - 1.0
    return go.Mesh3d(x=p[:, 2], y=p[:, 1], z=p[:, 0], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                     color="deepskyblue", opacity=0.45, name="fronte u = t")

fig = go.Figure(data=[ostacoli, linea, punti, fronte(istanti[0])],
                frames=[go.Frame(data=[fronte(t)], traces=[3], name=f"{t:.2f}") for t in istanti])
fig.update_layout(
    title="Ostacoli, percorso da A a B e fronte d'onda partito da B",
    scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube"),
    height=750, margin=dict(l=0, r=0, t=40, b=0),
    sliders=[dict(currentvalue=dict(prefix="fronte u = "), pad=dict(t=30),
                  steps=[dict(label=f"{t:.2f}", method="animate",
                              args=[[f"{t:.2f}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for t in istanti])])
fig.write_html("percorso_3d.html", include_plotlyjs="cdn")         # anche come file da scaricare: si apre nel browser
fig.show()

In [ ]:
# =====================================================================
# ANIMAZIONE: la linea del percorso cresce da A verso B (usa le variabili della cella precedente)
# =====================================================================
n_frame = 60
s_tot = s_perc[-1]                                                  # lunghezza del percorso
s_fitto = np.linspace(0.0, s_tot, 600)                              # 600 punti equidistanti lungo il percorso
P_fitto = np.stack([np.interp(s_fitto, s_perc, percorso[:, c]) for c in range(3)], axis=1)

def tratto(m):                                                      # linea da A fino al punto m e punta sul punto m
    return [go.Scatter3d(x=P_fitto[:m + 1, 0], y=P_fitto[:m + 1, 1], z=P_fitto[:m + 1, 2], mode="lines",
                         line=dict(color="darkred", width=9), name="percorso"),
            go.Scatter3d(x=[P_fitto[m, 0]], y=[P_fitto[m, 1]], z=[P_fitto[m, 2]], mode="markers",
                         marker=dict(size=6, color="orange"), name="drone")]

passi = np.linspace(0, len(s_fitto) - 1, n_frame).astype(int)       # indice del punto raggiunto in ogni frame
anim = go.Figure(data=[ostacoli, punti] + tratto(0),
                 frames=[go.Frame(data=tratto(m), traces=[2, 3], name=str(f)) for f, m in enumerate(passi)])
anim.update_layout(
    title="Percorso da A a B (premi Play)",
    scene=dict(xaxis=dict(range=[-1, 1]), yaxis=dict(range=[-1, 1]), zaxis=dict(range=[-1, 1]), aspectmode="cube"),
    height=750, margin=dict(l=0, r=0, t=40, b=80),
    updatemenus=[dict(type="buttons", direction="left", x=0.1, y=0, xanchor="right", yanchor="top", pad=dict(r=10, t=70), buttons=[
        dict(label="Play", method="animate",
             args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True, transition=dict(duration=0))]),
        dict(label="Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])])],
    sliders=[dict(x=0.1, len=0.9, y=0, yanchor="top", currentvalue=dict(prefix="percorso: "), pad=dict(t=50),
                  steps=[dict(label=f"{100 * s_fitto[m] / s_tot:.0f}%", method="animate",
                              args=[[str(f)], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for f, m in enumerate(passi)])])
anim.write_html("percorso_animato.html", include_plotlyjs="cdn")    # anche come file da scaricare: si apre nel browser
anim.show()